<a href="https://colab.research.google.com/github/bhardwaj-amit79/Colab/blob/main/%5BUpgrad_Mod_2%5D_Lec_1_Prompt_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Prompting Techniques
A basic walkthrough of:
- Section 0 - Setup
- Section 1 - Intro to Prompting
- Section 2 - Zero-Shot Prompting
- Section 3 - Few-Shot Prompting
- Section 4 - Chain-of-Thought Prompting
- Section 5 - Meta-Prompting
- Section 6 - Tree of Thoughts


## Section 0: Setup for local environment

In [ ]:
# Install required libraries
# !pip install transformers accelerate

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Section 1: Intro to Prompting

The 'transformers' library by HuggingFace provides us with various ways of interacting with LLMs. In this section we will look into the various ways of prompting a model using tokenizers or pipelines.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# We are going to use Llama 3.2
model_name = "unsloth/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

sample_text = "Hello world"
tokens = tokenizer(sample_text)
print(tokens)  # By default tokens are returned as a list, but the model expects tensors

tokens = tokenizer(sample_text, return_tensors="pt").to('cuda')
print(tokens)  # Model accepts this format

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

{'input_ids': [128000, 9906, 1917], 'attention_mask': [1, 1, 1]}
{'input_ids': tensor([[128000,   9906,   1917]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1]], device='cuda:0')}


In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
# device_map indicates where to load the model parameters
# device_map = None       -> whole model on GPU
# device_map = "auto"     -> spread across multiple GPUs automatically
# device_map = "balanced" -> evenly distributed across GPUs
# device_map = "cpu"      -> run on CPU (slower, but works everywhere)
# Note: device_map is intended for inference, not distributed training

output_tokens = model.generate(
                  input_ids=tokens["input_ids"],
                  attention_mask=tokens["attention_mask"],
                  max_new_tokens=100
                )
# A cleaner syntax: output_tokens = model.generate(**tokens, max_new_tokens=100)
print(output_tokens)  # Raw generated token IDs

result = tokenizer.decode(output_tokens[0])
print(result)  # Decoded human-readable text

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tensor([[128000,   9906,   1917,   2268,   2028,    374,    264,   1296,   1984,
            505,    856,  33195,  12827,    382,    334,   4727,   4076,  68063,
           1296,   5621,   5481,    198,    334,  18697,   4076,  68063,   1925,
            198,    334,  34541,   6668,  68063,  40122,   4513,    271,   2028,
            374,    264,   2082,  44165,    512,  74694,  12958,    198,    755,
          41886,   3232,     25,    610,      8,   1492,    610,    512,    262,
           4304,   5715,    264,  35649,  43213,   7275,    262,    471,    282,
              1,   9906,     11,    314,    609,     92,  25765,  14196,   4077,
          10267,    757,   1440,    422,    499,   1205,   4205,    775,   2268,
          45464,    334,   4126,    220,     16,     25,  57277,     40,   3077,
           1903,   1063,   4442,    311,    279,   2082,  44165,     13,   5810,
            596,    279,   6177,   2373]], device='cuda:0')
<|begin_of_text|>Hello world!

This is a test mes

In [ ]:
# Sampling parameters: temperature, top_k, top_p
output_tokens = model.generate(
    **tokens,
    temperature=0.7,   # higher = more creative, lower = more focused
    top_k=10,          # only consider the top-10 most likely next tokens
    top_p=0.9,         # nucleus sampling: keep tokens covering 90% probability mass
)
result = tokenizer.decode(output_tokens[0])
print(result)

<|begin_of_text|>Hello world!

I'm creating a simple program to calculate the sum of all even numbers up to a given number.

Here's my code:

```python
def sum_even_numbers(n):
    """
    Calculate the sum of all even numbers up to n.

    Args:
        n (int): The upper limit.

    Returns:
        int: The sum of all even numbers up to n.
    """
    sum_even = 0
    for i in range(n):
        if i % 2 == 0:
            sum_even += i
    return sum_even

# Example usage
n = 100
result = sum_even_numbers(n)
print(f"The sum of all even numbers up to {n} is: {result}")
```

This code works fine for small inputs, but it's not very efficient for large inputs. The reason is that it has a time complexity of O(n), which means it will take a long time to calculate the sum for large inputs.

How can I optimize this code to make it more efficient for large inputs?

### Step 1: Identify the problem
The problem with the current code is that it checks every number up to `n` to see if it's even. 

### Chat Structure
In **chat mode** the conversation is broken into turns, each assigned a **role**:

| Role | Meaning |
|------|---------|
| system | Global instructions that shape the model's behaviour |
| user | What the human says |
| assistant | What the model replies |


In [ ]:
# Example multi-turn chat
messages = [
    {"role": "system",    "content": "You are a helpful tutor."},
    {"role": "user",      "content": "What is Python?"},
    {"role": "assistant", "content": "Python is a programming language."},
    {"role": "user",      "content": "Who created it?"},
]

tokens = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
output_tokens = model.generate(**tokens, max_new_tokens=100)
result = tokenizer.decode(output_tokens[0])
print(result)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 23 May 2026

You are a helpful tutor.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is Python?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Python is a programming language.<|eot_id|><|start_header_id|>user<|end_header_id|>

Who created it?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Python was created by Guido van Rossum, a Dutch computer programmer. He started working on Python in the late 1980s and released the first version, Python 0.9.0, in 1991.<|eot_id|>


### Pipelines – the easiest way to run a model

In [ ]:
from transformers import pipeline

# Build a text-generation pipeline reusing the already-loaded model & tokenizer
model_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=500,
    temperature=0.1,
    top_k=10,
    top_p=0.85,
)

result = model_pipeline("The sky is")
print(result)

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'top_k', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "The sky is painted with hues of crimson and gold as the sun sets over the rolling hills of Tuscany. The air is filled with the sweet scent of blooming wildflowers and the soft chirping of crickets. It's a moment of perfect tranquility, a time to unwind and let the stresses of the world melt away.\n\nAs the sun dips lower in the sky, the light takes on a warm, golden glow, casting long shadows across the landscape. The hills, once a deep green, now seem to glow with an inner light, as if infused with a sense of peace and serenity. The trees, too, seem to be bathed in a soft, ethereal radiance, their leaves rustling softly in the gentle breeze.\n\nThe sky above is a kaleidoscope of colors, a swirling vortex of pinks, oranges, and purples, as the sun's final rays dance across the horizon. The stars begin to twinkle like diamonds in the darkening sky, and the world is bathed in a soft, lunar glow.\n\nAs the night deepens, the sounds of the world grow quieter, and the o

In [ ]:
# pipeline can also download and load a model by itself
model_pipeline2 = pipeline("text-generation", model="gpt2")
result = model_pipeline2("Once upon a time")
print(result)

In [ ]:
# Pipelines also handle the chat template automatically
messages = [
    {"role": "system", "content": "You are a helpful medical assistant chatbot."},
    {"role": "user",   "content": "How are you?"},
]

response = model_pipeline(messages)
print(response)

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'system', 'content': 'You are a helpful medical assistant chatbot.'}, {'role': 'user', 'content': 'How are you?'}, {'role': 'assistant', 'content': "I'm functioning properly, thank you for asking. I'm here to provide you with information and assistance with any medical-related questions or concerns you may have. How can I help you today?"}]}]


### Helper functions (used throughout all sections)

In [ ]:
def make_message(system_content, user_content):
    """Wrap system + user text into the chat-template format."""
    return [
        {"role": "system", "content": system_content},
        {"role": "user",   "content": user_content},
    ]

def fetch_response(response):
    """Extract the assistant reply text from a pipeline response."""
    return response[0]["generated_text"][-1]["content"]

## Section 2: Zero-Shot Prompting
**Zero-shot prompting** means asking the model to do a task *without* giving it any examples.
We rely entirely on the model's pre-trained knowledge and the instructions in the prompt.


In [ ]:
# Example 1 – Simple arithmetic
system_prompt = "You are a helpful math expert."

user_prompt = """
What is the result of: 128 + 256
Answer:
"""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


To find the result, I'll add 128 and 256:

128 + 256 = 384

The answer is 384.


In [ ]:
# Example 2 – Generate a property listing from a fact sheet
fact_sheet_house = """
OVERVIEW
- 4-bedroom, 3-bathroom single-family home in a quiet suburban neighbourhood.
- Built in 2015, renovated in 2022.

FEATURES
- Modern kitchen with granite countertops and stainless-steel appliances.
- Hardwood flooring in living areas; carpet in bedrooms.
- Master suite with walk-in closet and soaking tub.

LOCATION
- Sector 3, Saltlake, Kolkata, India
- Near top-rated schools, parks, and public transport.

DETAILS
- 2,450 sq ft interior | 6,000 sq ft lot
- HOA: Rs.50/month | Property tax: ~Rs.6,200/year
"""

system_prompt = "You are a real estate marketing assistant who writes compelling property descriptions."
user_prompt = "Create a property description based on the following fact sheet: ```" + fact_sheet_house + "```"

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


**Elegant Suburban Retreat in the Heart of Saltlake**

Escape to a serene oasis in the bustling city of Kolkata with this stunning 4-bedroom, 3-bathroom single-family home, perfectly situated in Sector 3, Saltlake. Built in 2015 and meticulously renovated in 2022, this property boasts a perfect blend of modern amenities and classic charm.

As you step inside, you'll be greeted by the sleek and sophisticated modern kitchen, complete with granite countertops and high-end stainless-steel appliances, perfect for culinary enthusiasts. The spacious living areas feature beautiful hardwood flooring, creating a warm and inviting atmosphere, while the bedrooms are carpeted for ultimate comfort.

The pièce de résistance is the master suite, a serene retreat that exudes luxury and relaxation. With a spacious walk-in closet and a soaking tub, you'll feel like you're in a spa every day. The home's thoughtful design ensures that every aspect of your daily life is made easier, from the convenient loca

In [ ]:
# Example 3 – Sentiment analysis (inferring facts from text)
headphones_review = """
I've been using these headphones for about two weeks now.
The sound quality is excellent: clear highs and strong bass.
They're super comfortable even after hours of use.
Battery life lasts me through multiple workdays.
Only downside is the case feels a bit cheap, but overall I'm very happy.
"""

system_prompt = "You are a sentiment analysis assistant. Determine whether a product review is positive or negative."

user_prompt = (
    "What is the sentiment of the following review (delimited by triple backticks)?\n"
    "Give your answer as a single word: positive or negative.\n\n"
    "Review: ```" + headphones_review + "```"
)

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


positive


## Section 3: Few-Shot Prompting
**Few-shot prompting** gives the model a small number of worked examples *inside the prompt* before asking the real question.
The examples act as a pattern the model can follow, making it much more reliable on structured or domain-specific tasks.

> **Rule of thumb:** 2–5 examples are usually enough. More examples → better accuracy, but also more tokens.


In [ ]:
# Example 1 – Classify movie reviews as POSITIVE / NEGATIVE
# We provide 3 labelled examples before asking about a new review.

system_prompt = "You are a movie review classifier. Classify reviews as POSITIVE or NEGATIVE."

user_prompt = """\
Here are some examples:

Review: "An absolute masterpiece. The acting was superb and the story kept me hooked."
Sentiment: POSITIVE

Review: "Terrible plot, wooden acting, and a predictable ending. Complete waste of time."
Sentiment: NEGATIVE

Review: "A beautiful film with stunning visuals and a touching story."
Sentiment: POSITIVE

Now classify this review:
Review: "The movie was boring and too long. I almost fell asleep in the second half."
Sentiment:\
"""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sentiment: NEGATIVE


In [ ]:
# Example 2 – Extract structured information from customer feedback
# The model learns the JSON output format from the examples.

system_prompt = "You are a data extraction assistant. Extract information from customer feedback into JSON."

user_prompt = """\
Here are some examples:

Feedback: "The delivery was super fast, arrived the next day! But the packaging was damaged."
Output: {"delivery": "fast", "packaging": "damaged", "overall_sentiment": "mixed"}

Feedback: "Product quality is amazing and the price is very reasonable."
Output: {"product_quality": "amazing", "price": "reasonable", "overall_sentiment": "positive"}

Now extract information from this feedback:
Feedback: "Customer support was very helpful, but the product stopped working after a week."
Output:\
"""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Here's the extracted information in JSON format:

{"customer_support": "helpful", "product": "stopped working", "duration": "a week", "overall_sentiment": "mixed"}


In [ ]:
# Example 3 – Text translation with a consistent format
# Useful when you need the model to follow a strict input/output pattern.

system_prompt = "You are a translation assistant. Translate English sentences to French."

user_prompt = """\
Here are some examples:

English: Good morning, how are you?
French: Bonjour, comment allez-vous?

English: The weather is beautiful today.
French: Le temps est magnifique aujourd'hui.

English: I would like a cup of coffee, please.
French: Je voudrais une tasse de cafe, s'il vous plait.

Now translate:
English: Where is the nearest hospital?
French:\
"""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The translation of the sentence "Where is the nearest hospital?" in French is:

Où est l'hôpital le plus proche?


## Section 4: Chain-of-Thought (CoT) Prompting
**Chain-of-Thought prompting** asks the model to *think step by step* before giving the final answer.
This dramatically improves accuracy on reasoning tasks (math, logic, multi-step problems).

**Two flavours:**
1. **Zero-shot CoT** – just add *"Let's think step by step."* to the prompt.
2. **Few-shot CoT** – provide examples that include the reasoning steps.


In [ ]:
# Zero-shot CoT
# The magic phrase "Let's think step by step" triggers step-by-step reasoning.

system_prompt = "You are a helpful math tutor."

user_prompt = """\
A store sells apples for Rs.15 each and oranges for Rs.20 each.
If Maya buys 4 apples and 3 oranges, how much does she spend in total?

Let's think step by step.\
"""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


To find out how much Maya spends in total, we need to calculate the cost of the apples and the cost of the oranges separately, and then add them together.

Step 1: Calculate the cost of the apples
Maya buys 4 apples, and each apple costs Rs.15. So, the total cost of the apples is:
4 apples * Rs.15/apple = Rs.60

Step 2: Calculate the cost of the oranges
Maya buys 3 oranges, and each orange costs Rs.20. So, the total cost of the oranges is:
3 oranges * Rs.20/orange = Rs.60

Step 3: Calculate the total amount Maya spends
Now, we add the cost of the apples and the cost of the oranges together:
Rs.60 (apples) + Rs.60 (oranges) = Rs.120

So, Maya spends a total of Rs.120.


In [ ]:
# Few-shot CoT
# Examples show the model *how* to reason, not just the final answer.

system_prompt = "You are a logical reasoning assistant. Always show your reasoning before the final answer."

user_prompt = """\
Here are some examples:

Question: A train travels at 60 km/h. How far does it go in 2.5 hours?
Reasoning:
  Step 1: Distance = Speed x Time
  Step 2: Distance = 60 x 2.5 = 150 km
Answer: 150 km

Question: A shop has 200 items. 40% are sold. How many remain?
Reasoning:
  Step 1: Items sold = 40% of 200 = 0.40 x 200 = 80
  Step 2: Items remaining = 200 - 80 = 120
Answer: 120 items

Now solve:
Question: A recipe needs 3 cups of flour for 12 cookies. How many cups are needed for 40 cookies?
Reasoning:\
"""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


To solve this problem, I will follow the same steps as before.

Question: A recipe needs 3 cups of flour for 12 cookies. How many cups are needed for 40 cookies?

Reasoning:
  Step 1: First, we need to find the ratio of cups of flour to cookies. This can be calculated by dividing the number of cups by the number of cookies.
  Step 2: Ratio = Cups of flour / Number of cookies = 3 / 12 = 0.25 cups per cookie
  Step 3: Now, we can multiply the ratio by the number of cookies we want to make to find the total cups of flour needed.
  Step 4: Cups of flour needed = Ratio x Number of cookies = 0.25 x 40 = 10 cups

Answer: 10 cups


In [ ]:
# CoT for logical / commonsense reasoning

system_prompt = "You are a logical reasoning assistant. Think step by step before answering."

user_prompt = """\
All mammals are warm-blooded.
Whales are mammals.
Dolphins are mammals.
Sharks are fish, not mammals.

Question: Are whales and dolphins warm-blooded? Is a shark warm-blooded?
Let's think step by step.\
"""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


To answer this question, let's analyze the information step by step:

1. All mammals are warm-blooded. This is a general statement that applies to all mammals.
2. Whales are mammals. Since all mammals are warm-blooded, whales must also be warm-blooded.
3. Dolphins are mammals. Again, since all mammals are warm-blooded, dolphins must also be warm-blooded.
4. Sharks are fish, not mammals. This statement tells us that sharks do not belong to the mammalian category, but it does not provide information about their temperature regulation.

From the information provided, we can conclude that whales and dolphins are warm-blooded, as they are mammals. However, we cannot determine the temperature regulation of sharks based on the given information. We only know that they are fish, not mammals.

So, the answer to the question is:

- Whales and dolphins are warm-blooded.
- We cannot determine if a shark is warm-blooded based on the given information.


## Section 5: Meta-Prompting
**Meta-prompting** uses the model to *generate or improve prompts* rather than directly answering a task.
Think of it as asking the model to be its own prompt engineer.

**Common use cases:**
- Let the model write a better prompt for a given task.
- Ask the model to reflect on its own output and revise it.
- Have the model plan the steps needed before executing them.


In [ ]:
# Example 1 – Ask the model to generate a good prompt for a task
system_prompt = (
    "You are an expert prompt engineer. "
    "When given a task description, write a clear, detailed prompt that another "
    "AI model could use to complete the task accurately."
)

user_prompt = """\
Task: I want an AI to help students understand a complex scientific concept
by using simple analogies and everyday examples.

Write a prompt for this task.\
"""

response = model_pipeline(make_message(system_prompt, user_prompt))
generated_prompt = fetch_response(response)
print("=== Generated Prompt ===")
print(generated_prompt)

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Generated Prompt ===
Here's a detailed prompt for an AI model to help students understand a complex scientific concept using simple analogies and everyday examples:

**Task Description:**

Create a clear and concise explanation of [Complex Scientific Concept] that is accessible to students of [Grade Level/Subject]. The explanation should utilize simple analogies and everyday examples to illustrate key concepts and make the material more relatable and understandable.

**Specific Requirements:**

1. **Conceptual Clarity:** The explanation should accurately convey the core principles and key concepts of [Complex Scientific Concept].
2. **Analogies and Examples:** Incorporate at least 3-5 simple analogies and everyday examples that illustrate the complex concept, making it more tangible and easier to grasp for students.
3. **Relatable Language:** Use simple, clear, and concise language that is free of technical jargon and complex terminology.
4. **Engaging Tone:** Write in an approacha

In [ ]:
# Example 2 – Self-refinement: ask the model to critique and improve its own output
system_prompt = "You are a writing assistant who helps improve text quality."

# Step 1: Get an initial draft
initial_user_prompt = "Write a short product description for a smartwatch."
response = model_pipeline(make_message(system_prompt, initial_user_prompt))
initial_output = fetch_response(response)
print("=== Initial Output ===")
print(initial_output)

# Step 2: Ask the model to critique and improve its own draft
refinement_prompt = (
    "Here is a product description:\n\n"
    + initial_output
    + "\n\nCritique this description: what is weak or missing?\n"
    "Then rewrite it to be more engaging and persuasive."
)

response = model_pipeline(make_message(system_prompt, refinement_prompt))
print("\n=== Refined Output ===")
print(fetch_response(response))

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Initial Output ===
Here's a short product description for a smartwatch:

**Introducing the Pulse Pro - Your Ultimate Wearable Companion**

Stay connected, track your fitness, and control your world with the Pulse Pro smartwatch. This sleek and feature-rich timepiece combines the latest in wearable technology with a stylish design, ensuring you're always on top of your game.

**Key Features:**

- Advanced heart rate monitoring and fitness tracking
- GPS, Bluetooth, and Wi-Fi connectivity for seamless connectivity
- Compatible with both iOS and Android devices
- Long-lasting battery life up to 5 days
- Water-resistant up to 50 meters
- Customizable watch faces and notifications

**Upgrade your daily routine with the Pulse Pro. Order now and experience the future of wearable technology.**

=== Refined Output ===
Critique:

1. The description is a bit generic and doesn't provide a clear unique selling point (USP) for the Pulse Pro smartwatch.
2. The list of features is straightforward 

In [ ]:
# Example 3 – Task planning: model generates a plan, then executes it
system_prompt = (
    "You are a project planning assistant. "
    "First create a numbered step-by-step plan, then execute each step."
)

user_prompt = """\
Task: Write a beginner's guide to starting a vegetable garden at home.

First, list the steps you will cover. Then write the guide.\
"""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


**Step-by-Step Plan:**

1. Planning and Preparation
2. Choosing the Right Location
3. Preparing the Soil
4. Selecting the Right Vegetables
5. Preparing the Garden Bed
6. Planting the Vegetables
7. Watering and Maintenance
8. Pest and Disease Management
9. Harvesting and Enjoying

**Beginner's Guide to Starting a Vegetable Garden at Home**

**Step 1: Planning and Preparation**

Starting a vegetable garden can be a rewarding experience, but it requires some planning and preparation. Before you begin, consider the following:

* Determine the amount of space you have available for your garden.
* Decide which vegetables you want to grow and how much space they require.
* Check the climate and weather conditions in your area to determine which vegetables will thrive.
* Make a list of the necessary tools and materials you'll need to get started.

**Step 2: Choosing the Right Location**

When choosing a location for your vegetable garden, consider the following factors:

* Sunlight: Most veget

## Section 6: **[Try it yourself]** Tree of Thoughts (ToT)
**Tree of Thoughts** extends Chain-of-Thought by exploring *multiple reasoning paths* in parallel and selecting the best one.
Instead of committing to a single line of thinking, the model proposes several candidate "thoughts", evaluates them, and continues from the most promising branch.

**Simplified 3-step pattern (all in one prompt):**
1. **Explore** – generate several different solution approaches.
2. **Evaluate** – score or critique each approach.
3. **Select & solve** – pick the best approach and complete the solution.

> Full ToT normally uses search algorithms (BFS/DFS) with multiple model calls.
> Here we simulate the key idea in a single prompt to keep things beginner-friendly.


In [ ]:
# Example 1 – Business strategy problem
system_prompt = (
    "You are a strategic business advisor. "
    "Use the Tree of Thoughts method:\n"
    "  1. Generate THREE different approaches to solve the problem.\n"
    "  2. Briefly evaluate the pros and cons of each.\n"
    "  3. Select the best approach and explain the solution in detail."
)

user_prompt = """\
Problem: A small local bakery wants to increase its revenue by 30%
within the next 6 months without significantly increasing costs.

Apply Tree of Thoughts reasoning.\
"""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


**Tree of Thoughts: Increasing Revenue by 30% without Significant Cost Increase**

**Initial Thoughts:**

* Increase revenue by 30% in 6 months
* No significant cost increase
* Small local bakery

**Branch 1: Diversify Product Offerings**

* Introduce new products or flavors to existing menu
* Partner with local suppliers to source unique ingredients
* Offer catering services or delivery options

**Pros:**

* Potential for increased revenue through new product sales
* Ability to differentiate the bakery from competitors
* Opportunities for partnerships and collaborations

**Cons:**

* Additional investment in product development and marketing
* Risk of cannibalizing existing sales with new products
* Potential for increased labor costs if hiring additional staff

**Branch 2: Expand Online Presence**

* Develop a strong e-commerce platform for online sales
* Utilize social media and email marketing to promote products
* Offer discounts and promotions to online customers

**Pros:**

* Ab

In [ ]:
# Example 2 – Logical puzzle
system_prompt = (
    "You are a logical reasoning assistant. "
    "Use Tree of Thoughts:\n"
    "  1. List THREE different reasoning paths or interpretations.\n"
    "  2. Evaluate which path is consistent with all the given facts.\n"
    "  3. Choose the correct path and give the final answer."
)

user_prompt = """\
Puzzle:
- There are three boxes: Red, Blue, and Green.
- One box contains gold, one contains silver, and one is empty.
- The label on the Red box says "Gold".
- The label on the Blue box says "Empty".
- The label on the Green box says "Silver".
- ALL labels are wrong.

Where is the gold?

Apply Tree of Thoughts reasoning.\
"""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


To solve this puzzle using Tree of Thoughts, I'll list three different reasoning paths or interpretations:

**Path 1:**
- The label on the Red box says "Gold", but it's actually empty.
- The label on the Blue box says "Empty", but it's actually gold.
- The label on the Green box says "Silver", but it's actually gold.

**Path 2:**
- The label on the Red box says "Gold", but it's actually silver.
- The label on the Blue box says "Empty", but it's actually gold.
- The label on the Green box says "Silver", but it's actually empty.

**Path 3:**
- The label on the Red box says "Gold", but it's actually silver.
- The label on the Blue box says "Empty", but it's actually empty.
- The label on the Green box says "Silver", but it's actually gold.

Now, let's evaluate which path is consistent with all the given facts:

- The label on the Red box says "Gold", but it's actually empty. (Path 1 and Path 3)
- The label on the Blue box says "Empty", but it's actually gold. (Path 1)
- The label on the G

In [ ]:
# Example 3 – Creative writing with ToT
system_prompt = (
    "You are a creative writing coach. "
    "Use Tree of Thoughts:\n"
    "  1. Brainstorm THREE different story opening approaches.\n"
    "  2. Evaluate each for impact, originality, and engagement.\n"
    "  3. Select the best and write a full opening paragraph."
)

user_prompt = """\
Write an opening paragraph for a short story about a scientist
who discovers that time is running backwards.

Apply Tree of Thoughts reasoning.\
"""

response = model_pipeline(make_message(system_prompt, user_prompt))
print(fetch_response(response))

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


**Brainstorming THREE different story opening approaches:**

1. **The Ordinary Day**: Start with a mundane scene of the scientist going about their daily routine, only to have the reader realize that something is off as they notice objects and events unfolding in reverse.
2. **The Mysterious Discovery**: Begin with the scientist stumbling upon an unusual phenomenon, such as a clock or a calendar, that hints at the reversal of time.
3. **The Personal Crisis**: Introduce the scientist in a moment of personal crisis, such as a failed experiment or a lost loved one, and then have them experience a strange and inexplicable reversal of time.

**Evaluating each approach:**

1. **The Ordinary Day**: This approach has a high potential for impact, as it allows the reader to become familiar with the scientist's daily life before the twist is revealed. However, it may lack originality, as the "ordinary day" trope is common in science fiction. Engagement is moderate, as the reader will be curious a

---
## Summary

| Technique | Core Idea | Best For |
|-----------|-----------|----------|
| **Zero-Shot** | No examples – rely on instructions alone | Simple, well-defined tasks |
| **Few-Shot** | Provide 2–5 worked examples | Structured outputs, domain tasks |
| **Chain-of-Thought** | Ask the model to reason step by step | Math, logic, multi-step problems |
| **Meta-Prompting** | Use the model to generate/improve prompts | Prompt design, self-refinement, planning |
| **Tree of Thoughts** | Explore multiple reasoning paths, pick the best | Complex decisions, creative tasks, puzzles |
